In [16]:
import pandas as pd

books = pd.read_csv("../data/books_with_categories.csv")

In [17]:
from transformers import pipeline

# classifier for Eckman 6: anger, disgust, fear, joy, neutral, sadness, surprise
classifier = pipeline("text-classification",
                      model="j-hartmann/emotion-english-distilroberta-base",
                      top_k = None,
                      device = "mps")
classifier("I love this!")

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

[[{'label': 'joy', 'score': 0.9771687984466553},
  {'label': 'surprise', 'score': 0.008528684265911579},
  {'label': 'neutral', 'score': 0.005764600355178118},
  {'label': 'anger', 'score': 0.004419781267642975},
  {'label': 'sadness', 'score': 0.002092392183840275},
  {'label': 'disgust', 'score': 0.001611993182450533},
  {'label': 'fear', 'score': 0.0004138521908316761}]]

In [18]:
books["description"][0]

'Brit Bennett’s chart topping novel, The Vanishing Half, is a story that tracks the lives of twin African American twin sisters who, after witnessing the murder of their father, run away at age 16. One sister begins passing as white and the other sister remains true to her identity. The Vanishing Half explores the intricacies of identity, family, and race in a provocative, but compassionate way.'

In [19]:
classifier(books["description"][0])
# not a very accurate classification

[[{'label': 'neutral', 'score': 0.48212823271751404},
  {'label': 'anger', 'score': 0.1903698742389679},
  {'label': 'disgust', 'score': 0.18022051453590393},
  {'label': 'sadness', 'score': 0.10230214148759842},
  {'label': 'joy', 'score': 0.030334845185279846},
  {'label': 'fear', 'score': 0.011391024105250835},
  {'label': 'surprise', 'score': 0.003253332106396556}]]

In [20]:
# classify sentences individually
classifier(books["description"][0].split("."))

[[{'label': 'sadness', 'score': 0.7779311537742615},
  {'label': 'fear', 'score': 0.09698846191167831},
  {'label': 'neutral', 'score': 0.057081520557403564},
  {'label': 'anger', 'score': 0.03138400614261627},
  {'label': 'disgust', 'score': 0.02546197734773159},
  {'label': 'surprise', 'score': 0.006792923901230097},
  {'label': 'joy', 'score': 0.00435994565486908}],
 [{'label': 'neutral', 'score': 0.6222614049911499},
  {'label': 'disgust', 'score': 0.27846527099609375},
  {'label': 'anger', 'score': 0.03981650248169899},
  {'label': 'sadness', 'score': 0.019346458837389946},
  {'label': 'fear', 'score': 0.01727641187608242},
  {'label': 'surprise', 'score': 0.017251966521143913},
  {'label': 'joy', 'score': 0.005581961944699287}],
 [{'label': 'neutral', 'score': 0.6325600147247314},
  {'label': 'disgust', 'score': 0.13805851340293884},
  {'label': 'joy', 'score': 0.10447536408901215},
  {'label': 'anger', 'score': 0.09951731562614441},
  {'label': 'sadness', 'score': 0.012589156627

In [21]:
sentences = books["description"][0].split(".")
predictions = classifier(sentences)
sorted(predictions[0], key=lambda x: x["label"])
# take the sentence with the highest probability for each sentiment

[{'label': 'anger', 'score': 0.03138400614261627},
 {'label': 'disgust', 'score': 0.02546197734773159},
 {'label': 'fear', 'score': 0.09698846191167831},
 {'label': 'joy', 'score': 0.00435994565486908},
 {'label': 'neutral', 'score': 0.057081520557403564},
 {'label': 'sadness', 'score': 0.7779311537742615},
 {'label': 'surprise', 'score': 0.006792923901230097}]

In [22]:
import numpy as np

emotion_labels = ["anger", "disgust", "fear", "joy", "sadness", "surprise", "neutral"]
isbn = []
emotion_scores = {label: [] for label in emotion_labels}

# creates a dictionary for each description containing the maximumm probability for each emotion
def calculate_max_emotion_scores(predictions):
    per_emotion_scores = {label: [] for label in emotion_labels}
    for prediction in predictions:
        sorted_predictions = sorted(prediction, key=lambda x: x["label"])
        for index, label in enumerate(emotion_labels):
            per_emotion_scores[label].append(sorted_predictions[index]["score"])
    return {label: np.max(scores) for label, scores in per_emotion_scores.items()}

In [23]:
# test for the first 10 books
for i in range(10):
    isbn.append(books["isbn13"][i])
    sentences = books["description"][i].split(".")
    predictions = classifier(sentences)
    max_scores = calculate_max_emotion_scores(predictions)
    for label in emotion_labels:
        emotion_scores[label].append(max_scores[label])

In [24]:
emotion_scores

{'anger': [np.float64(0.09951731562614441),
  np.float64(0.15472441911697388),
  np.float64(0.0849376767873764),
  np.float64(0.012789599597454071),
  np.float64(0.06413350999355316),
  np.float64(0.06413350999355316),
  np.float64(0.14891961216926575),
  np.float64(0.06413350999355316),
  np.float64(0.014368480071425438),
  np.float64(0.71426922082901)],
 'disgust': [np.float64(0.27846527099609375),
  np.float64(0.16712960600852966),
  np.float64(0.523088276386261),
  np.float64(0.0094162467867136),
  np.float64(0.42624256014823914),
  np.float64(0.10400652140378952),
  np.float64(0.3405281603336334),
  np.float64(0.10400652140378952),
  np.float64(0.01741030625998974),
  np.float64(0.49667873978614807)],
 'fear': [np.float64(0.09698846191167831),
  np.float64(0.9890568852424622),
  np.float64(0.06028831750154495),
  np.float64(0.9519777297973633),
  np.float64(0.1222962886095047),
  np.float64(0.05136270076036453),
  np.float64(0.3652559518814087),
  np.float64(0.41607916355133057),


In [26]:
from tqdm import tqdm
import re

emotion_labels = ["anger", "disgust", "fear", "joy", "sadness", "surprise", "neutral"]
isbn = []
emotion_scores = {label: [] for label in emotion_labels}

for i in tqdm(range(len(books))):
    isbn.append(books["isbn13"][i])
    sentences = re.split(r'[.?!]+', books["description"][i])
    predictions = classifier(sentences)
    max_scores = calculate_max_emotion_scores(predictions)
    for label in emotion_labels:
        emotion_scores[label].append(max_scores[label])

100%|██████████| 5000/5000 [04:15<00:00, 19.58it/s]


In [27]:
emotions_df = pd.DataFrame(emotion_scores)
emotions_df["isbn13"] = isbn

In [28]:
emotions_df

,anger,disgust,fear,joy,sadness,surprise,neutral,isbn13
0,0.099517,0.278465,0.096988,0.104475,0.632560,0.777931,0.078765,9780349701462
1,0.154724,0.167130,0.989057,0.234059,0.549477,0.963760,0.261492,9780062853509
2,0.084938,0.523088,0.060288,0.109274,0.944272,0.199291,0.107244,9780567688682
3,0.064134,0.104007,0.965474,0.764379,0.549477,0.111690,0.078765,9780807507896
4,0.064134,0.426243,0.122296,0.835246,0.930934,0.968373,0.340749,9781944316174
...,...,...,...,...,...,...,...,...
4995,0.084938,0.139802,0.024546,0.014326,0.955254,0.046409,0.270131,9784906962860
4996,0.214713,0.738636,0.427887,0.935053,0.711036,0.825442,0.382379,9781420128840
4997,0.185730,0.215533,0.154925,0.896459,0.579213,0.273537,0.131470,9781504058582
4998,0.127801,0.762259,0.148509,0.102071,0.925001,0.111690,0.539080,9789492051448


In [29]:
# add the maximum score for each emotion to the book vectors
books = pd.merge(books, emotions_df, on = "isbn13")

In [30]:
books

,isbn13,title,authors,categories,description,thumbnail,published_year,average_rating,num_pages,title_and_subtitle,tagged_description,simple_categories,anger,disgust,fear,joy,sadness,surprise,neutral
0,9780349701462,Vanishing Half,Brit Bennett,African American;Twins;fiction;race;identity;c...,"Brit Bennett’s chart topping novel, The Vanish...",https://covers.openlibrary.org/b/id/10680969-L...,2020,NaN,352.0,Vanishing Half: A Novel,9780349701462 Brit Bennett’s chart topping nov...,Fiction,0.099517,0.278465,0.096988,0.104475,0.632560,0.777931,0.078765
1,9780062853509,Ghost Radio,Leopoldo Gout,Fiction;Radio broadcasters in fiction;Radio ta...,Ghost Radio is a terrifying novel about a ghos...,NaN,2018,NaN,NaN,Ghost Radio: A Novel,9780062853509 Ghost Radio is a terrifying nove...,Fiction,0.154724,0.167130,0.989057,0.234059,0.549477,0.963760,0.261492
2,9780567688682,Exodus 1-18,Graham I. Davies;Graham I. Davies;Christopher ...,"Bible, commentaries, o. t. pentateuch;Bible;Co...","""Graham I. Davies provides his long-awaited co...",NaN,2020,NaN,816.0,Exodus 1-18: A Critical and Exegetical Commentary,"9780567688682 ""Graham I. Davies provides his l...",Nonfiction,0.084938,0.523088,0.060288,0.109274,0.944272,0.199291,0.107244
3,9780807507896,Skeleton Key Mystery,Gertrude Chandler Warner;Anthony VanArsdale,"Adventure and adventurers, fiction;Brothers an...",The Aldens are visiting a small town known for...,https://covers.openlibrary.org/b/id/13770184-L...,2020,NaN,128.0,Skeleton Key Mystery,9780807507896 The Aldens are visiting a small ...,Children's Fiction,0.064134,0.104007,0.965474,0.764379,0.549477,0.111690,0.078765
4,9781944316174,Matthew Wong,Matthew Wong;Cheim & Read,Exhibitions;Art,"Over the course of his brief career, Matthew W...",NaN,2021,NaN,NaN,Matthew Wong: footprints in the wind : ink dra...,9781944316174 Over the course of his brief car...,Nonfiction,0.064134,0.426243,0.122296,0.835246,0.930934,0.968373,0.340749
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4995,9784906962860,Environmental teachings for the anthropocene,Atsushi Nobayashi;Scott Simon,Taiwan aborigines;Ethnobotany;Ethnozoology;Mus...,"""The essays in this volume are organised into ...",NaN,2020,NaN,230.0,Environmental teachings for the anthropocene: ...,"9784906962860 ""The essays in this volume are o...",Nonfiction,0.084938,0.139802,0.024546,0.014326,0.955254,0.046409,0.270131
4996,9781420128840,"Night, Sea and Stars",Heather Graham,Fiction;Aircraft accidents;Large type books;Fi...,Crash-landing on a remote South Pacific island...,NaN,2020,NaN,352.0,"Night, Sea and Stars",9781420128840 Crash-landing on a remote South ...,Fiction,0.214713,0.738636,0.427887,0.935053,0.711036,0.825442,0.382379
4997,9781504058582,Nine Lives to Murder,Marian Babson,Actors;Cats;Fiction;Metamorphosis;Large type b...,"Marion Babson always seems to write funny, lig...",NaN,2019,NaN,NaN,Nine Lives to Murder,9781504058582 Marion Babson always seems to wr...,Fiction,0.185730,0.215533,0.154925,0.896459,0.579213,0.273537,0.131470
4998,9789492051448,Zilverbeek,Lucas Leffler,Artistic Photography;Pictorial works,Since the 1920s the Belgian factory Gevaert ac...,NaN,2019,NaN,NaN,Zilverbeek: Silver Creek,9789492051448 Since the 1920s the Belgian fact...,Nonfiction,0.127801,0.762259,0.148509,0.102071,0.925001,0.111690,0.539080


In [31]:
books.to_csv("../data/books_with_emotions.csv", index = False)